# 03 Model 1: Word Embeddings + Classifier
Training of architecture 1: frozen pretrained word embeddings with a 2-layer classifier (ReLU non-linearity). Only the classifier is trained. Experiments are tracked with Weights & Biases.

Load the features and labels that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only used once at the end, in `05_evaluation.ipynb`.

In [1]:
import numpy as np
import torch

data = np.load("features/model1.npz")
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)

torch.Size([16801, 1200]) torch.float32
torch.Size([16801]) torch.int64


## Classifier
The 2-layer classifier like in the course exercise "PyTorch" (Week 1): linear layer, ReLU, linear layer. Input: 1200 features, output: 2 classes (h1 or h2).

In [2]:
import torch.nn as nn

class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
print(model)

Classifier(
  (linear1): Linear(in_features=1200, out_features=100, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=100, out_features=2, bias=True)
)


Number of parameters per layer (weights and biases), as in the course exercise "PyTorch" (Week 1).

In [3]:
for name, params in model.named_parameters():
    print(name, params.shape)

linear1.weight torch.Size([100, 1200])
linear1.bias torch.Size([100])
linear2.weight torch.Size([2, 100])
linear2.bias torch.Size([2])


Test the untrained classifier on the first 5 training examples: one pair of scores (h1, h2) per example.

In [4]:
x = X_train[:5]
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 1200])
torch.Size([5, 2])
tensor([[-0.1848, -0.1143],
        [-0.1771, -0.0636],
        [-0.1528, -0.0544],
        [-0.1209, -0.0621],
        [-0.1661, -0.0300]], grad_fn=<AddmmBackward0>)


## Loss
Cross-entropy loss for the 2 classes (course slides "Preliminaries", slide 27). `nn.CrossEntropyLoss` applies the softmax to the raw scores itself, so the classifier has no softmax layer. Before training, the loss should be about -log(0.5) ≈ 0.69, the value of random guessing between 2 classes.

In [5]:
criterion = nn.CrossEntropyLoss()

y = y_train[:5]
loss = criterion(output, y)
print(y)
print(loss)

tensor([1, 1, 0, 0, 1])
tensor(0.6781, grad_fn=<NllLossBackward0>)


## One training step 
The Adam optimizer updates the weights (course slides "Preliminaries", slide 29). One training step on the 5 examples as in the course exercise "PyTorch" (Week 1): forward pass, loss, backpropagation, update. Afterwards, the loss on the same 5 examples should be lower.

In [6]:
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=0.001)

output = model(x)
loss = criterion(output, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(loss)
print(criterion(model(x), y))

tensor(0.6781, grad_fn=<NllLossBackward0>)
tensor(0.6114, grad_fn=<NllLossBackward0>)


## Batches
The data is split into batches with a `DataLoader`, as in the course exercise "Experiment Tracking": training data in batches of 32, shuffled in every epoch. 16801 = 525 x 32 + 1, so the last batch would contain a single example; `drop_last=True` skips it (a different example in every epoch, because of the shuffling). For measuring loss and accuracy, all batches have the same size, so that the average of the batch losses equals the average over all examples: validation data in 10 batches of 100, training data in 53 batches of 317 (16801 = 53 x 317), both not shuffled. `TensorDataset` pairs each row of features with its label.

In [7]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True, drop_last=True)
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

x_batch, y_batch = next(iter(train_loader))
print(len(train_loader), len(train_eval_loader), len(valid_loader))
print(x_batch.shape, y_batch.shape)

525 53 10
torch.Size([32, 1200]) torch.Size([32])


## Evaluation
Loss and accuracy on a dataset, batch by batch, as in the validation of the course exercise "Experiment Tracking": the loss of each batch and the predictions (`argmax`, the class with the higher score) and labels are collected; at the end, the loss is the average of the batch losses and the accuracy is the share of correct predictions over all examples. `model.eval()` and `torch.no_grad()` as in the `test_loop` of the PyTorch tutorial "Optimizing Model Parameters". A new untrained model is tested first: it should be at about 0.69 loss and 50% accuracy.

In [8]:
def evaluate(model, loader):
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            losses.append(criterion(output, y).item())
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6943, accuracy: 49.10%


## Experiment tracking
The training runs are tracked with Weights and Biases (W&B), as in the course exercise "Experiment Tracking". Log in to W&B (the API key was saved once with `wandb login` in the terminal).

In [9]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\mmici\_netrc.
wandb: Currently logged in as: maria-micioni (maria-micioni-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


True

## Training
Training loop as in the PyTorch tutorial "Optimizing Model Parameters": in every epoch, one training step (as in the course exercise "PyTorch") for each batch of the training data, then evaluation on the training and validation data. The training is written as a function `train()` with the hyperparameters as arguments, as in the course exercise "Experiment Tracking", so that it can be run with different hyperparameters. Every call trains a fresh model with seed 42 and creates one W&B run. Each run is logged to W&B: the hyperparameters in `config` and, after every epoch, the training and validation loss and accuracy with `wandb.log` (with `epoch` for the x-axis). `wandb.finish()` ends the run. After every epoch, the model is saved to `checkpoints/` if its validation loss is the lowest so far (`torch.save` of the `state_dict`, as in the PyTorch tutorial "Save and Load the Model"). At the end, the weights of the best epoch are loaded again, so `train()` returns the best model, not the one after the last epoch. The validation loss is used because it changes more smoothly than the accuracy.

In [10]:
import os

os.makedirs("checkpoints", exist_ok=True)

def train(inner_dim=100, learning_rate=0.001, num_epochs=10):
    name = f"glove-mlp-h{inner_dim}-lr{learning_rate}"
    checkpoint_path = f"checkpoints/{name}.pth"
    wandb.init(
        project="nlp-project1-art",
        name=name,
        config={
            "model": "glove-mlp",
            "inner_dim": inner_dim,
            "learning_rate": learning_rate,
            "epochs": num_epochs,
            "batch_size": train_loader.batch_size,
            "seed": 42,
        },
    )

    torch.manual_seed(42)
    model = Classifier(1200, 2, inner_dim=inner_dim)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)

    wandb.finish()
    print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}")
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return model

model = train(inner_dim=100, learning_rate=0.001)

Epoch 1: train loss 0.6693, train acc 60.63%, valid loss 0.6751, valid acc 58.70%
Epoch 2: train loss 0.6449, train acc 62.09%, valid loss 0.6654, valid acc 57.30%
Epoch 3: train loss 0.6251, train acc 64.97%, valid loss 0.6576, valid acc 60.50%
Epoch 4: train loss 0.6187, train acc 65.10%, valid loss 0.6590, valid acc 59.60%
Epoch 5: train loss 0.6051, train acc 66.71%, valid loss 0.6511, valid acc 61.50%
Epoch 6: train loss 0.6058, train acc 66.32%, valid loss 0.6654, valid acc 59.90%
Epoch 7: train loss 0.5844, train acc 68.74%, valid loss 0.6530, valid acc 62.20%
Epoch 8: train loss 0.5737, train acc 69.86%, valid loss 0.6494, valid acc 62.70%
Epoch 9: train loss 0.5630, train acc 70.32%, valid loss 0.6572, valid acc 60.20%
Epoch 10: train loss 0.5458, train acc 72.52%, valid loss 0.6500, valid acc 63.30%


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▂▄▄▅▄▆▆▇█
train_loss,█▇▅▅▄▄▃▃▂▁
val_acc,▃▁▅▄▆▄▇▇▄█
val_loss,█▅▃▄▁▅▂▁▃▁
epoch,10
train_acc,0.72519
train_loss,0.54578
val_acc,0.633
val_loss,0.65001


Best epoch: 8, valid loss 0.6494


Check that the returned model is the one of the best epoch: its validation loss and accuracy must be the same as in that epoch.

In [11]:
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6494, accuracy: 62.70%


## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier`, based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Parameters and test forward pass: `named_parameters()` and a forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss: cross-entropy from the course slides "Preliminaries" (Week 1), slide 27; `criterion` and `loss = criterion(output, target)` as in the course exercise "PyTorch" (Week 1); `nn.CrossEntropyLoss` (applies the softmax to the raw scores) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html.
- Random seed: fixed seed 42 as in the course exercise "Experiment Tracking" (`seed_everything(42)`); `torch.manual_seed` from the PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html.
- Optimizer and training step: Adam from the course slides "Preliminaries" (Week 1), slide 29; `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` as in the course exercise "PyTorch" (Week 1).
- Batches: `DataLoader` with `batch_size=32` and `shuffle=True` for training and no shuffling for validation as in the course exercise "Experiment Tracking"; `TensorDataset` and `drop_last=True` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html.
- Evaluation: collecting the batch losses, `argmax` predictions and labels, joining them with `torch.cat` and averaging the batch losses as in the validation of the course exercise "Experiment Tracking" (`validation_step` and `on_validation_epoch_end`); `model.eval()` and `torch.no_grad()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters", https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html; `np.mean` from NumPy (Harris et al., 2020).
- Training loop: epochs with a train loop over all batches and an evaluation after each epoch, and `model.train()`, as in the PyTorch tutorial "Optimizing Model Parameters"; the training step inside as in the course exercise "PyTorch" (Week 1); training as a function `train()` with the hyperparameters as arguments (with default values) and the seed set inside, as in the course exercise "Experiment Tracking".
- Experiment tracking: `wandb.login()`, `wandb.init(project=..., config=...)` and `wandb.finish()` as in the course exercise "Experiment Tracking"; run name `name=` from the W&B `wandb.init()` reference, https://docs.wandb.ai/models/ref/python/functions/init; logging the metrics once per epoch with `epoch` as x-axis as in the W&B documentation "Log objects and media", https://docs.wandb.ai/models/track/log (written there as `run.log`, which is the same as `wandb.log` for the current run); `train_loader.batch_size` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html; Weights and Biases (Biewald, 2020); `nbformat` installed for W&B, as in the course exercise "Experiment Tracking".
- Saving the best model: `torch.save(model.state_dict(), ...)` and `model.load_state_dict(torch.load(..., weights_only=True))` as in the PyTorch tutorial "Save and Load the Model", https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html; `os.makedirs` from the Python standard library, https://docs.python.org/3/library/os.html; `float("inf")` from the Python documentation, https://docs.python.org/3/library/functions.html#float.
- Tools: code written with the help of Claude (Anthropic); all code reviewed and understood by me.